# Day 15 Fine-tuning Demo (Synthetic Data)

This notebook is a lightweight, self-contained fine-tuning demo based on `resources/fine_tune_sample.ipynb`.

We generate a small synthetic dataset (30 examples) and run a supervised fine-tune so you can see the full workflow.


## Prerequisites

- An `OPENAI_API_KEY` in your environment or `.env`
- The `openai` Python package installed
- Expect API usage costs when you run the fine-tune


In [ ]:
import json
import os
import random
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

random.seed(42)

load_dotenv(override=True)
UPLOAD_API_KEY = os.getenv("OPENAI_UPLOAD_API_KEY")
client = OpenAI()
fine_tune_client = OpenAI(api_key=UPLOAD_API_KEY) if UPLOAD_API_KEY else client

print("Client ready.")


## Step 1: Create a small synthetic dataset

We will build a simple support-ticket classifier with 3 labels: `billing`, `technical`, and `shipping`.
The output is strict: the label only.


In [ ]:
SYSTEM_PROMPT = (
    "You are a support triage assistant. "
    "Classify the ticket into one of: billing, technical, shipping. "
    "Respond with only the label."
)

label_templates = {
    "billing": [
        "I was charged twice for my subscription this month.",
        "My invoice total is wrong and includes extra fees.",
        "Please refund the duplicate payment on my card.",
        "I need a copy of my receipt for last week's payment.",
        "The discount code did not apply to my bill.",
        "Can you update my billing address on file?",
        "Why did the price increase on my renewal?",
        "My payment failed but my card is valid.",
        "I want to change from annual to monthly billing.",
        "There is an unexpected charge on my account.",
    ],
    "technical": [
        "The app crashes every time I open the dashboard.",
        "I cannot log in even after resetting my password.",
        "File uploads are stuck at 0%.",
        "The API returns a 500 error for all requests.",
        "I'm seeing a blank screen after the latest update.",
        "Notifications are not showing up on my phone.",
        "The integration stopped syncing with our CRM.",
        "Two-factor authentication codes are not arriving.",
        "Search results are empty even for known items.",
        "The report export never finishes downloading.",
    ],
    "shipping": [
        "My order says delivered but I never received it.",
        "Can you change the delivery address for my package?",
        "The tracking link hasn't updated in three days.",
        "Please expedite shipping for order #5521.",
        "I received the wrong item in my shipment.",
        "The package arrived damaged and torn.",
        "When will my backordered item ship?",
        "I want to cancel the shipment before it goes out.",
        "The courier left my package at the wrong door.",
        "Can I switch to pickup instead of delivery?",
    ],
}

examples = []
for label, texts in label_templates.items():
    for text in texts:
        examples.append(
            {
                "messages": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": text},
                    {"role": "assistant", "content": label},
                ]
            }
        )

random.shuffle(examples)
len(examples)


In [ ]:
train_size = 24
train_examples = examples[:train_size]
val_examples = examples[train_size:]

len(train_examples), len(val_examples)


In [ ]:
data_dir = Path("resources/fine_tune_demo_data")
data_dir.mkdir(parents=True, exist_ok=True)


def write_jsonl(path: Path, rows: list[dict]) -> None:
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row) + "\n")


train_path = data_dir / "train.jsonl"
val_path = data_dir / "val.jsonl"

write_jsonl(train_path, train_examples)
write_jsonl(val_path, val_examples)

train_path, val_path


## Step 2: Upload the JSONL files

Each file must be uploaded with purpose `fine-tune`.


In [ ]:
with open(train_path, "rb") as f:
    train_file = fine_tune_client.files.create(file=f, purpose="fine-tune")

with open(val_path, "rb") as f:
    val_file = fine_tune_client.files.create(file=f, purpose="fine-tune")

train_file, val_file


## Step 2.5: Baseline Response (Before Fine-tuning)

Capture a baseline response from the base model so you can compare before vs after.


In [ ]:
BASE_MODEL = "gpt-4.1-nano-2025-04-14"

baseline_ticket = "I was charged twice for my last invoice. Please refund the duplicate charge."

baseline_response = fine_tune_client.responses.create(
    model=BASE_MODEL,
    input=baseline_ticket,
)

print(baseline_response.output_text)


## Step 3: Create a fine-tuning job

Choose a supported base model. Replace `BASE_MODEL` if needed.


In [ ]:
job = fine_tune_client.fine_tuning.jobs.create(
    model=BASE_MODEL,
    training_file=train_file.id,
    validation_file=val_file.id,
)

job


## Step 4: Monitor the job


In [ ]:
job_id = job.id
fine_tune_client.fine_tuning.jobs.retrieve(job_id)


In [ ]:
fine_tune_client.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10)


## Step 5: Test the Fine-tuned Model (After Fine-tuning)

When the job finishes, use the returned `fine_tuned_model` id for inference.


In [ ]:
fine_tuned_model = fine_tune_client.fine_tuning.jobs.retrieve(job_id).fine_tuned_model
fine_tuned_model


In [ ]:
test_ticket = "I was charged twice for my last invoice. Please refund the duplicate charge."

response = fine_tune_client.responses.create(
    model=fine_tuned_model,
    input=test_ticket,
)

print(response.output_text)
